# 🛒 Example 1 — E-commerce Checkout A/B Test

**Scenario:** An e-commerce team tested a redesigned checkout flow (treatment)
against the existing flow (control). The hypothesis is that reducing friction
at checkout will lift conversion rate and revenue per user.

**Experiment design:**
- 50/50 random split
- 7-day run, ~10,000 users
- Primary metric: `conversion_rate` (binary)
- Secondary metric: `revenue_per_user` (continuous, right-skewed)
- Pre-experiment covariates: `account_age_days`, `device_type`, `country_tier`

**What we'll check:**
1. Was the split as expected? (SRM)
2. Were the arms balanced on pre-experiment covariates?
3. Did the new checkout lift conversion and revenue?
4. Are the results robust to bootstrap?

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd

from ab_diagnostics import (
    run_ab_diagnostics,
    print_ab_report,
    make_config,
)

In [ ]:
# ── Synthetic e-commerce dataset ──────────────────────────────────────────────
# Replace this cell with: df = pd.read_csv('your_experiment.csv')

def make_ecommerce_data(n=10_000, seed=7):
    rng = np.random.default_rng(seed)
    group  = rng.choice(['control','treatment'], n, p=[0.5, 0.5])
    W      = (group == 'treatment').astype(int)

    # Pre-experiment covariates
    account_age_days = rng.exponential(300, n).clip(1, 2000).round(0)
    device_type      = rng.choice(['mobile','desktop','tablet'], n, p=[0.6, 0.32, 0.08])
    country_tier     = rng.choice(['tier1','tier2','tier3'], n, p=[0.35, 0.40, 0.25])

    # Outcomes — new checkout reduces friction: +3pp conversion, +$1.80 revenue
    conv_prob   = (0.08 + 0.03 * W
                   + 0.02 * (device_type == 'desktop')
                   + 0.01 * (country_tier == 'tier1')).clip(0, 1)
    converted   = rng.binomial(1, conv_prob)

    # Revenue is zero for non-converters, log-normal for converters
    base_rev      = np.where(converted == 1,
                             np.exp(rng.normal(3.5, 0.8, n)), 0.0)
    revenue       = (base_rev + 1.80 * W * converted).clip(0).round(2)

    return pd.DataFrame(dict(
        user_id          = np.arange(n),
        group            = group,
        converted        = converted,
        revenue          = revenue,
        account_age_days = account_age_days,
        device_type      = device_type,
        country_tier     = country_tier,
    ))


df = make_ecommerce_data(seed=7)
print(f"Dataset: {len(df):,} users")
print()
print(df.groupby('group')[['converted','revenue','account_age_days']]
      .agg(['mean','std']).round(3).to_string())

---
## Step 1 — Configure the diagnostic

Two things to customise in `make_config()`:
- Column names to match your DataFrame schema
- The `expected_split` matching your experiment's randomisation ratio

In [ ]:
cfg = make_config(
    treatment_col   = 'group',
    treatment_val   = 'treatment',
    control_val     = 'control',
    expected_split  = 0.5,
    alpha           = 0.05,
    srm_tolerance   = 0.01,
    n_bootstrap     = 2000,
    bootstrap_seed  = 42,
)
print("Config:", cfg)

---
## Step 2 — Run the full pipeline

In [ ]:
results = run_ab_diagnostics(
    df,
    continuous_metrics     = ['revenue'],
    binary_metrics         = ['converted'],
    continuous_covariates  = ['account_age_days'],
    categorical_covariates = ['device_type', 'country_tier'],
    run_bootstrap          = True,
    stop_on_srm            = True,
    config                 = cfg,
)

print_ab_report(results)

---
## Step 3 — Drill into findings programmatically

The `results` dict gives you full access to every finding for custom analysis,
dashboards, or downstream alerting.

In [ ]:
# ── Pull out specific findings ─────────────────────────────────────────────────
print("=== Effect Findings ===")
for f in results['effects']:
    d = f.get('detail', {})
    if 'abs_lift' in d:
        print(f"  {f['code']}")
        print(f"    Abs lift : {d['abs_lift']:+.4f}")
        print(f"    Rel lift : {d['rel_lift']:+.1%}")
        print(f"    95% CI   : [{d['ci_low']:.4f}, {d['ci_high']:.4f}]")
        print(f"    p-value  : {f['pvalue']:.4f}")
    elif 'ate' in d:
        print(f"  {f['code']}")
        print(f"    ATE      : {d['ate']:+.4f}")
        print(f"    Rel lift : {d['relative_lift']:+.1%}")
        print(f"    95% CI   : [{d['ci_low']:.4f}, {d['ci_high']:.4f}]")
        print(f"    p-value  : {f['pvalue']:.4f}")

print()
print("=== Bootstrap Findings ===")
for f in results['bootstrap']:
    d = f.get('detail', {})
    print(f"  {f['code']}: observed={d.get('observed',0):+.4f}, "
          f"CI=[{d.get('ci_low',0):.4f}, {d.get('ci_high',0):.4f}]")

In [ ]:
# ── Summary: how many fails / warns? ──────────────────────────────────────────
from collections import Counter

all_findings = [f for section in results.values() for f in section]
counts = Counter(f['level'] for f in all_findings)
print(f"Total findings: {len(all_findings)}")
print(f"  ✔ ok   : {counts['ok']}")
print(f"  ⚠ warn : {counts['warn']}")
print(f"  ✘ fail : {counts['fail']}")
print()

# Any fails?
fails = [f for f in all_findings if f['level'] == 'fail']
if fails:
    print("FAIL findings:")
    for f in fails:
        print(f"  [{f['code']}] {f['message']}")
else:
    print("✅ No FAIL findings — experiment looks clean.")

---
## Step 4 — Sensitivity: what if the effect is driven by mobile users?

Run diagnostics on a subgroup. Note: multiple subgroup tests inflate Type I error —
use this for exploration, not primary analysis.

In [ ]:
# ── Subgroup: mobile users only ────────────────────────────────────────────────
df_mobile = df[df['device_type'] == 'mobile'].copy()
print(f"Mobile users: {len(df_mobile):,} ({len(df_mobile)/len(df):.0%} of total)")
print()

results_mobile = run_ab_diagnostics(
    df_mobile,
    continuous_metrics     = ['revenue'],
    binary_metrics         = ['converted'],
    continuous_covariates  = ['account_age_days'],
    categorical_covariates = ['country_tier'],
    run_bootstrap          = False,   # faster for exploration
    config                 = cfg,
)
print_ab_report(results_mobile)